In [1]:
%run utils.py

In [2]:
spark = get_spark_session(app_name="Streaming")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/02 10:34:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/02 10:34:25 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession started with app name: Streaming


## public.orders -> bronze.orders

In [3]:
from pyspark.sql.types import *
from pyspark.sql.functions import col, from_json

cdc_schema = StructType([
    StructField("payload", StructType([
        StructField("after", StructType([
            StructField("id", IntegerType(), True),
            StructField("product_name", StringType(), True),
            StructField("status", StringType(), True)
        ]), True),
        StructField("op", StringType(), True),
        StructField("ts_ms", LongType(), True)
    ]), True)
])

# Read the stream from Kafka
df_kafka = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "host.docker.internal:9092") \
    .option("subscribe", "dbserver1.public.orders") \
    .option("startingOffsets", "earliest") \
    .load()

df_parsed = df_kafka.select(
    from_json(col("value").cast("string"), cdc_schema).alias("data")
).select(
    col("data.payload.after.id").cast("string").alias("id"),
    col("data.payload.after.product_name").alias("product_name"),
    col("data.payload.after.status").alias("status"),
    col("data.payload.op").alias("op"),
    col("data.payload.ts_ms").alias("ts_ms")
).filter(col("id").isNotNull()) # Hudi yêu cầu khóa chính không được null


table_name_cow = "orders"
bronze = "bronze"
base_path = f"s3a://{bronze}/cdc_postgres/public/orders"
checkpointLocation = f"s3a://{bronze}/cdc_postgres/public/orders/checkpoints"

cow_hudi_conf = {
    "hoodie.table.name": table_name_cow, # The name of our Hudi table.
    "hoodie.database.name": bronze,
    "hoodie.datasource.write.recordkey.field": "id", # The column that acts as the unique identifier for each record.
    "hoodie.datasource.write.table.type": "COPY_ON_WRITE", # Hudi uses Copy-on-Write as the default table type, but we are being explicit here.
    "hoodie.datasource.write.partitionpath.field": "status", # The column Hudi uses to partition the data on storage.
    "hoodie.datasource.write.precombine.field": "ts_ms", # The field used to deduplicate records when a conflict occurs.
    "hoodie.table.cdc.enabled": "true",
    "hoodie.datasource.write.hive_style_partitioning": "true", # This ensures partition directories are named like `city=new_york`.
    "hoodie.datasource.hive_sync.mode": "hms",
    "hoodie.datasource.hive_sync.jdbcurl": "thrift://hive-metastore:9083",
    "hoodie.datasource.hive_sync.enable": "true",
    "hoodie.datasource.hive_sync.support_timestamp": "true",
    'hoodie.upsert.shuffle.parallelism': 2,
    'hoodie.insert.shuffle.parallelism': 2
}

# write stream to new hudi table
stream_query = df_parsed.writeStream.format("hudi") \
    .options(**cow_hudi_conf) \
    .outputMode("append") \
    .option("path", f"{base_path}/{table_name_cow}") \
    .option("checkpointLocation", checkpointLocation) \
    .trigger(processingTime='3 seconds') \
    .start()


# WARNING: Unable to get Instrumentation. Dynamic Attach failed. You may add this JAR as -javaagent manually, or supply -Djdk.attach.allowAttachSelf
# WARNING: Unable to attach Serviceability Agent. sun.jvm.hotspot.memory.Universe.getNarrowOopBase()


## CRBT.test_cdc -> broze.test_cdc

In [5]:
from pyspark.sql.types import *
from pyspark.sql.functions import col, from_json

cdc_schema = StructType([
    StructField("payload", StructType([
        StructField("after", StructType([
            StructField("id", IntegerType(), True),
            StructField("product_name", StringType(), True),
            StructField("status", StringType(), True)
        ]), True),
        StructField("op", StringType(), True),
        StructField("ts_ms", LongType(), True)
    ]), True)
])

# Read the stream from Kafka
df_kafka = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "host.docker.internal:9092") \
    .option("subscribe", "dbserver1.CRBT.test_cdc") \
    .option("startingOffsets", "earliest") \
    .load()

df_parsed = df_kafka.select(
    from_json(col("value").cast("string"), cdc_schema).alias("data")
).select(
    col("data.payload.after.id").cast("string").alias("id"),
    col("data.payload.after.product_name").alias("product_name"),
    col("data.payload.after.status").alias("status"),
    col("data.payload.op").alias("op"),
    col("data.payload.ts_ms").alias("ts_ms")
).filter(col("id").isNotNull()) # Hudi yêu cầu khóa chính không được null


table_name_cow = "test_cdc"
bronze = "bronze"
base_path = f"s3a://{bronze}/cdc_postgres/CRBT/{table_name_cow}"
checkpointLocation = f"s3a://{bronze}/cdc_postgres/CRBT/{table_name_cow}/checkpoints"

cow_hudi_conf = {
    "hoodie.table.name": table_name_cow, # The name of our Hudi table.
    "hoodie.database.name": bronze,
    "hoodie.datasource.write.recordkey.field": "id", # The column that acts as the unique identifier for each record.
    "hoodie.datasource.write.table.type": "COPY_ON_WRITE", # Hudi uses Copy-on-Write as the default table type, but we are being explicit here.
    "hoodie.datasource.write.partitionpath.field": "status", # The column Hudi uses to partition the data on storage.
    "hoodie.datasource.write.precombine.field": "ts_ms", # The field used to deduplicate records when a conflict occurs.
    "hoodie.table.cdc.enabled": "true",
    "hoodie.datasource.write.hive_style_partitioning": "true", # This ensures partition directories are named like `city=new_york`.
    "hoodie.datasource.hive_sync.mode": "hms",
    "hoodie.datasource.hive_sync.jdbcurl": "thrift://hive-metastore:9083",
    "hoodie.datasource.hive_sync.enable": "true",
    "hoodie.datasource.hive_sync.support_timestamp": "true",
    'hoodie.upsert.shuffle.parallelism': 2,
    'hoodie.insert.shuffle.parallelism': 2
}

# write stream to new hudi table
stream_query = df_parsed.writeStream.format("hudi") \
    .options(**cow_hudi_conf) \
    .outputMode("append") \
    .option("path", f"{base_path}/{table_name_cow}") \
    .option("checkpointLocation", checkpointLocation) \
    .trigger(processingTime='3 seconds') \
    .start()


## CRBT.sub_collection_log -> bronze.sub_collection_log

In [6]:
from pyspark.sql.types import *
from pyspark.sql.functions import col, from_json

cdc_schema = StructType([
    StructField("payload", StructType([
        StructField("after", StructType([
            StructField("msisdn", StringType(), True),
            StructField("tone_id", StringType(), True),
            StructField("TYPE", IntegerType(), True),
            StructField("is_copy", IntegerType(), True),
            StructField("is_gift", IntegerType(), True),
            StructField("channel_id", IntegerType(), True),
            StructField("RESULT", IntegerType(), True),
            StructField("create_date", TimestampType(), True),
            StructField("ad_user", StringType(), True),
            StructField("sms_syntax", StringType(), True),
            StructField("description", StringType(), True)
        ]), True),
        StructField("op", StringType(), True),
        StructField("ts_ms", LongType(), True)
    ]), True)
])

# Read the stream from Kafka
df_kafka = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "host.docker.internal:9092") \
    .option("subscribe", "dbserver1.CRBT.sub_collection_log") \
    .option("startingOffsets", "earliest") \
    .load()

df_parsed = df_kafka.select(
    from_json(col("value").cast("string"), cdc_schema).alias("data")
).select(
    col("data.payload.after.msisdn").alias("msisdn"),
    col("data.payload.after.tone_id").alias("tone_id"),
    col("data.payload.after.TYPE").cast("string").alias("type"),
    col("data.payload.after.is_copy").cast("string").alias("is_copy"),
    col("data.payload.after.is_gift").cast("string").alias("is_gift"),
    col("data.payload.after.channel_id").cast("string").alias("channel_id"),
    col("data.payload.after.RESULT").cast("string").alias("result"),
    col("data.payload.after.create_date").alias("create_date"),
    col("data.payload.after.ad_user").alias("ad_user"),
    col("data.payload.after.sms_syntax").alias("sms_syntax"),
    col("data.payload.after.description").alias("description"),
    col("data.payload.op").alias("op"),
    col("data.payload.ts_ms").alias("ts_ms")
).filter(col("msisdn").isNotNull()) # Hudi yêu cầu khóa chính không được null


table_name_cow = "sub_collection_log"
bronze = "bronze"
base_path = f"s3a://{bronze}/cdc_postgres/CRBT/{table_name_cow}"
checkpointLocation = f"s3a://{bronze}/cdc_postgres/CRBT/{table_name_cow}/checkpoints"

cow_hudi_conf = {
    "hoodie.table.name": table_name_cow,
    "hoodie.database.name": bronze,
    "hoodie.datasource.write.recordkey.field": "msisdn,tone_id,create_date", # Composite primary key
    "hoodie.datasource.write.table.type": "COPY_ON_WRITE",
    "hoodie.datasource.write.partitionpath.field": "type", # Partition by TYPE
    "hoodie.datasource.write.precombine.field": "ts_ms",
    "hoodie.table.cdc.enabled": "true",
    "hoodie.datasource.write.hive_style_partitioning": "true",
    "hoodie.datasource.hive_sync.mode": "hms",
    "hoodie.datasource.hive_sync.jdbcurl": "thrift://hive-metastore:9083",
    "hoodie.datasource.hive_sync.enable": "true",
    "hoodie.datasource.hive_sync.support_timestamp": "true",
    'hoodie.upsert.shuffle.parallelism': 2,
    'hoodie.insert.shuffle.parallelism': 2
}

# write stream to new hudi table
stream_query = df_parsed.writeStream.format("hudi") \
    .options(**cow_hudi_conf) \
    .outputMode("append") \
    .option("path", f"{base_path}/{table_name_cow}") \
    .option("checkpointLocation", checkpointLocation) \
    .trigger(processingTime='3 seconds') \
    .start()

26/04/02 11:06:00 ERROR MicroBatchExecution: Query [id = 8aa25e3d-3526-49dd-a9e1-4d1876c0d803, runId = 3abdea88-c983-4a25-b4d7-646b0161db05] terminated with error
java.util.concurrent.ExecutionException: org.apache.kafka.common.errors.UnknownTopicOrPartitionException: This server does not host this topic-partition.
	at java.base/java.util.concurrent.CompletableFuture.reportGet(CompletableFuture.java:396)
	at java.base/java.util.concurrent.CompletableFuture.get(CompletableFuture.java:2073)
	at org.apache.kafka.common.internals.KafkaFutureImpl.get(KafkaFutureImpl.java:165)
	at org.apache.spark.sql.kafka010.ConsumerStrategy.retrieveAllPartitions(ConsumerStrategy.scala:66)
	at org.apache.spark.sql.kafka010.ConsumerStrategy.retrieveAllPartitions$(ConsumerStrategy.scala:65)
	at org.apache.spark.sql.kafka010.SubscribeStrategy.retrieveAllPartitions(ConsumerStrategy.scala:102)
	at org.apache.spark.sql.kafka010.SubscribeStrategy.assignedTopicPartitions(ConsumerStrategy.scala:113)
	at org.apache.

In [4]:
spark.stop()